In [1]:
"""
09_cac_summary.py
-------------------
Phase 11 prep: builds the channel-level summary table the
dashboard will use - average predicted CLV, an assumed CAC
(since real Amex CAC figures aren't public), and the resulting
CLV:CAC ratio that drives the targeting recommendations.

IMPORTANT: assumed_cac values below are illustrative assumptions,
clearly labeled as such - not real Amex figures. They're set in a
realistic relative order (paid channels cost more than organic/
referral), which is standard, defensible practice when real cost
data isn't available.

Reads:
    data/processed/segmented_customers.csv

Writes:
    data/processed/channel_cac_summary.csv
"""

import pandas as pd

df = pd.read_csv("data/processed/segmented_customers.csv")

# Illustrative, clearly-labeled assumptions - NOT real Amex data
ASSUMED_CAC = {
    "Paid Search": 950,
    "Social": 700,
    "Email": 350,
    "Organic": 150,
    "Referral": 100,
}

summary = df.groupby("acquisition_channel").agg(
    customer_count=("customer_id", "count"),
    avg_predicted_clv=("predicted_clv", "mean"),
).reset_index()

summary["assumed_cac"] = summary["acquisition_channel"].map(ASSUMED_CAC)
summary["clv_cac_ratio"] = (summary["avg_predicted_clv"] / summary["assumed_cac"]).round(1)
summary["avg_predicted_clv"] = summary["avg_predicted_clv"].round(2)
summary = summary.sort_values("clv_cac_ratio", ascending=False)

print("Channel summary (this feeds the dashboard's CAC vs CLV view):")
print(summary.to_string(index=False))

summary.to_csv("data/processed/channel_cac_summary.csv", index=False)
print("\nSaved: data/processed/channel_cac_summary.csv")

Channel summary (this feeds the dashboard's CAC vs CLV view):
acquisition_channel  customer_count  avg_predicted_clv  assumed_cac  clv_cac_ratio
           Referral             398           11927.50          100          119.3
            Organic             569            4472.20          150           29.8
              Email             783            4084.36          350           11.7
        Paid Search            1468            3453.27          950            3.6
             Social             782            2328.59          700            3.3

Saved: data/processed/channel_cac_summary.csv
